# Streaming
To provide a better user experience when calling LLMs we can use streaming to update the UI with the current state of our application or to display tokens as they are generated by the model.

With LangChain we have a few streaming modes available:
- **Stream agent progress**
- **Stream LLM tokens**
- **Stream custom updates**
- **Stream multiple nodes**

In [44]:
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
import requests

load_dotenv()

True

### Agent Progress
For streaming agent progrees, we need to use `stream_mode="updates"` and instead of invoking the agent with `.invoke()`, we use `.stream()`. This will produce chunks from which we can extract the needed data to send to the frontend.

Let's look at an example of an agent that uses a tool to fetch a joke from an API and streams progress updates.

<small>To learn more about tools, check [tools.ipynb](./tools.ipynb)</small>.

#### Create the Tool

In [10]:
URL='https://api.chucknorris.io/jokes/random'

@tool
def get_random_joke():
    """Fetches a random joke from an API"""
    
    response = requests.get(URL)

    if not response.ok:
        print(response.text)
        raise Exception('Something went wrong getting joke...')

    data = response.json()
    return data['value']

#### Create the Agent

In [11]:
joke_agent = create_agent(
    model='gpt-4o-mini',
    tools=[get_random_joke]
)

Let's first invoke our agent as we would normally without streaming to see the result.

In [14]:
result = joke_agent.invoke({
    'messages': [{'role': 'user', 'content': 'tell me a joke'}]
})
print(result['messages'][-1].content)

Here's a joke for you: 

In an average living room, there are 1,242 objects Chuck Norris could use to kill you, including the room itself.


#### Stream Agent Progress

Now we will stream the progress by using `agent.stream()` and `stream_mode='updates'` and print the chunks as they are returned from the model.

In [32]:
for chunk in joke_agent.stream(
    {'messages': [{'role': 'user', 'content': 'tell me a joke'}]}
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content}\n")

step: model
content: 

step: tools
content: Curators at Madame Tussauds Wax Museum had to remove the life size wax figure of Chuck Norris one day after they first installed it. It seems they found Chuck Norris' wax foot deeply inbedded in the face of the Michael Jackson figure.

step: model
content: Here's a joke for you:

Curators at Madame Tussauds Wax Museum had to remove the life-size wax figure of Chuck Norris one day after they first installed it. It seems they found Chuck Norris' wax foot deeply embedded in the face of the Michael Jackson figure.



We received three messages showing the updates. 
1. `AIMessage` - the message was a **tool call** which we can see from `'finish_reason': 'tool_calls'` and `tool_calls` list including the `get_random_joke` tool
2. `ToolMessage` - the tool fetched a joke and sent it to the model as a tool message.
3. `AIMessage` - the model combined our original query with the joke to generate a response

### Streaming LLM Tokens

Now let's stream each token as it is produced by the model. We will use the agent with `stream_mode` set to `messages`.

In [ ]:
tokens_agent = create_agent(
    model='gpt-4o-mini',
    tools=[get_random_joke]
)

In [45]:
for token, metadata in tokens_agent.stream(
    {'messages': [{'role': 'user', 'content': 'tell me a joke'}]},
    stream_mode='messages'
):
    if metadata['langgraph_step'] == 3:
        print(token.content)


Here's
 a
 joke
 for
 you
:
 Chuck
 Norris
 knows
 how
 to
 get
 to
 Sesame
 Street
.





As the model output each token, it was printed in its own `print()` function. Now we can send those tokens to the frontend (via websockets or SSE) and render the output as if it was typed in real time.